# 05 — Normalization Layers from Scratch

**Goal:** Implement BatchNorm, LayerNorm, GroupNorm, and RMSNorm from scratch (forward + backward), understand what each normalizes over, and know when to use which.

---

## The Core Idea

All normalization layers do the same thing over different axes:

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

where $\mu, \sigma^2$ are the mean/variance computed over specific dimensions, and $\gamma, \beta$ are learnable scale/shift parameters.

| Method | Normalizes over | Typical use |
|--------|----------------|-------------|
| **BatchNorm** | batch + spatial dims (per channel) | CNNs |
| **LayerNorm** | feature dims (per sample) | Transformers |
| **GroupNorm** | groups of channels (per sample) | CNNs with small batches |
| **RMSNorm** | feature dims, no centering | LLMs (LLaMA, etc.) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
%matplotlib inline

## 1. What Each Norm Normalizes — Diagram

Consider a tensor of shape `(N, C, H, W)` — batch, channels, height, width.

```
Tensor: (N, C, H, W)

BatchNorm:  compute mean/var across (N, H, W) for EACH channel C
            => mu, var have shape (C,)
            => same normalization for all samples in batch

LayerNorm:  compute mean/var across (C, H, W) for EACH sample N
            => mu, var have shape (N,)
            => each sample normalized independently

GroupNorm:  divide C channels into G groups, normalize across (C/G, H, W) per sample
            => mu, var have shape (N, G)

RMSNorm:    like LayerNorm but no mean subtraction, only scale by RMS
            => simpler, works well for transformers
```

In [ ]:
# Visual diagram of normalization axes
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

titles = ['BatchNorm', 'LayerNorm', 'InstanceNorm', 'GroupNorm (G=2)']
# For a (N=4, C=4) tensor (simplified 2D view)
N, C = 4, 4

for idx, (ax, title) in enumerate(zip(axes, titles)):
    grid = np.zeros((N, C))
    
    if idx == 0:  # BatchNorm: each column (channel) normalized together
        for c in range(C):
            grid[:, c] = c + 1
    elif idx == 1:  # LayerNorm: each row (sample) normalized together
        for n in range(N):
            grid[n, :] = n + 1
    elif idx == 2:  # InstanceNorm: each cell independently
        for n in range(N):
            for c in range(C):
                grid[n, c] = n * C + c + 1
    elif idx == 3:  # GroupNorm (G=2): groups of 2 channels per sample
        for n in range(N):
            grid[n, :2] = n * 2 + 1
            grid[n, 2:] = n * 2 + 2
    
    ax.imshow(grid, cmap='Set3', aspect='equal')
    ax.set_xlabel('Channels (C)')
    ax.set_ylabel('Batch (N)')
    ax.set_title(title)
    ax.set_xticks(range(C))
    ax.set_yticks(range(N))
    
    # Add text annotations
    for n in range(N):
        for c in range(C):
            ax.text(c, n, f'{grid[n,c]:.0f}', ha='center', va='center', fontsize=10)

plt.suptitle('Same color = normalized together (same mean/var computation)', y=1.03, fontsize=12)
plt.tight_layout()
plt.show()

## 2. Batch Normalization

**Training:** $\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$, then $y = \gamma \hat{x} + \beta$

**Inference:** Use running mean/variance instead of batch statistics.

**Backward pass (for 2D input of shape (N, D)):**
$$\frac{\partial L}{\partial \gamma} = \sum_i \frac{\partial L}{\partial y_i} \hat{x}_i$$
$$\frac{\partial L}{\partial \beta} = \sum_i \frac{\partial L}{\partial y_i}$$
$$\frac{\partial L}{\partial x_i} = \frac{\gamma}{\sqrt{\sigma^2 + \epsilon}} \left( \frac{\partial L}{\partial y_i} - \frac{1}{N}\sum_j \frac{\partial L}{\partial y_j} - \frac{\hat{x}_i}{N} \sum_j \frac{\partial L}{\partial y_j} \hat{x}_j \right)$$

In [ ]:
class BatchNorm:
    """Batch Normalization for fully connected layers (input shape: N x D)."""
    
    def __init__(self, num_features, momentum=0.1, eps=1e-5):
        self.gamma = np.ones((1, num_features))    # scale
        self.beta = np.zeros((1, num_features))     # shift
        self.eps = eps
        self.momentum = momentum
        self.training = True
        
        # Running stats for inference
        self.running_mean = np.zeros((1, num_features))
        self.running_var = np.ones((1, num_features))
        
        # Gradient storage
        self.dgamma = np.zeros_like(self.gamma)
        self.dbeta = np.zeros_like(self.beta)
    
    def forward(self, x):
        if self.training:
            # Batch statistics
            self.mu = np.mean(x, axis=0, keepdims=True)       # (1, D)
            self.var = np.var(x, axis=0, keepdims=True)        # (1, D)
            
            # Update running stats
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * self.mu
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * self.var
            
            # Normalize
            self.x_hat = (x - self.mu) / np.sqrt(self.var + self.eps)  # (N, D)
        else:
            # Use running stats
            self.x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
        
        self.x = x
        self.N = x.shape[0]
        return self.gamma * self.x_hat + self.beta
    
    def backward(self, dout):
        """Backward pass — the efficient formulation."""
        N = self.N
        x_hat = self.x_hat
        
        # Gradients for gamma and beta
        self.dgamma = np.sum(dout * x_hat, axis=0, keepdims=True)
        self.dbeta = np.sum(dout, axis=0, keepdims=True)
        
        # Gradient for input (efficient formula)
        dx_hat = dout * self.gamma
        std_inv = 1.0 / np.sqrt(self.var + self.eps)
        
        dx = std_inv / N * (
            N * dx_hat 
            - np.sum(dx_hat, axis=0, keepdims=True) 
            - x_hat * np.sum(dx_hat * x_hat, axis=0, keepdims=True)
        )
        return dx
    
    def parameters(self):
        return [(self.gamma, self.dgamma), (self.beta, self.dbeta)]


# Test
bn = BatchNorm(4)
x = np.random.randn(8, 4) * 3 + 5  # mean~5, std~3
out = bn.forward(x)
print(f"Input  mean: {x.mean(axis=0).round(2)}, var: {x.var(axis=0).round(2)}")
print(f"Output mean: {out.mean(axis=0).round(2)}, var: {out.var(axis=0).round(2)}")
print("After BN: mean ~ 0, var ~ 1 (before gamma/beta learned)")

## 3. Layer Normalization

Key difference from BatchNorm: normalizes across **features** (not batch). Each sample is normalized independently.

$$\mu_i = \frac{1}{D} \sum_{j=1}^{D} x_{i,j}, \quad \sigma_i^2 = \frac{1}{D} \sum_{j=1}^{D} (x_{i,j} - \mu_i)^2$$

In [ ]:
class LayerNorm:
    """Layer Normalization (normalizes across features for each sample)."""
    
    def __init__(self, normalized_shape, eps=1e-5):
        self.gamma = np.ones(normalized_shape)
        self.beta = np.zeros(normalized_shape)
        self.eps = eps
        self.dgamma = np.zeros_like(self.gamma)
        self.dbeta = np.zeros_like(self.beta)
    
    def forward(self, x):
        self.x = x
        self.mu = np.mean(x, axis=-1, keepdims=True)         # (N, 1)
        self.var = np.var(x, axis=-1, keepdims=True)          # (N, 1)
        self.std_inv = 1.0 / np.sqrt(self.var + self.eps)
        self.x_hat = (x - self.mu) * self.std_inv             # (N, D)
        return self.gamma * self.x_hat + self.beta
    
    def backward(self, dout):
        D = self.x.shape[-1]
        x_hat = self.x_hat
        
        self.dgamma = np.sum(dout * x_hat, axis=0)
        self.dbeta = np.sum(dout, axis=0)
        
        dx_hat = dout * self.gamma
        
        dx = self.std_inv / D * (
            D * dx_hat
            - np.sum(dx_hat, axis=-1, keepdims=True)
            - x_hat * np.sum(dx_hat * x_hat, axis=-1, keepdims=True)
        )
        return dx
    
    def parameters(self):
        return [(self.gamma, self.dgamma), (self.beta, self.dbeta)]


# Test
ln = LayerNorm(4)
x = np.random.randn(8, 4) * 3 + 5
out = ln.forward(x)
print(f"Per-sample mean of output: {out.mean(axis=-1).round(4)}")
print(f"Per-sample var of output:  {out.var(axis=-1).round(4)}")
print("Each sample independently has mean~0, var~1.")

## 4. Group Normalization

In [ ]:
class GroupNorm:
    """Group Normalization: divide channels into G groups, normalize within each group per sample."""
    
    def __init__(self, num_groups, num_channels, eps=1e-5):
        assert num_channels % num_groups == 0
        self.G = num_groups
        self.C = num_channels
        self.eps = eps
        self.gamma = np.ones((1, num_channels))
        self.beta = np.zeros((1, num_channels))
        self.dgamma = np.zeros_like(self.gamma)
        self.dbeta = np.zeros_like(self.beta)
    
    def forward(self, x):
        """x shape: (N, C) for FC or (N, C, H, W) for conv."""
        N = x.shape[0]
        self.x_shape = x.shape
        
        # Reshape to (N, G, C//G) for FC
        x_grouped = x.reshape(N, self.G, -1)
        self.mu = np.mean(x_grouped, axis=-1, keepdims=True)
        self.var = np.var(x_grouped, axis=-1, keepdims=True)
        self.std_inv = 1.0 / np.sqrt(self.var + self.eps)
        self.x_hat_grouped = (x_grouped - self.mu) * self.std_inv
        self.x_hat = self.x_hat_grouped.reshape(x.shape)
        
        return self.gamma * self.x_hat + self.beta
    
    def backward(self, dout):
        N = dout.shape[0]
        
        self.dgamma = np.sum(dout * self.x_hat, axis=0, keepdims=True)
        self.dbeta = np.sum(dout, axis=0, keepdims=True)
        
        dx_hat = (dout * self.gamma).reshape(N, self.G, -1)
        x_hat_g = self.x_hat_grouped
        D = dx_hat.shape[-1]  # channels per group
        
        dx_grouped = self.std_inv / D * (
            D * dx_hat
            - np.sum(dx_hat, axis=-1, keepdims=True)
            - x_hat_g * np.sum(dx_hat * x_hat_g, axis=-1, keepdims=True)
        )
        return dx_grouped.reshape(self.x_shape)
    
    def parameters(self):
        return [(self.gamma, self.dgamma), (self.beta, self.dbeta)]


# Test
gn = GroupNorm(num_groups=2, num_channels=8)
x = np.random.randn(4, 8) * 3 + 5
out = gn.forward(x)
# Check: within each group, mean~0, var~1
out_grouped = out.reshape(4, 2, 4)
print(f"Group means: {out_grouped.mean(axis=-1).round(3)}")
print(f"Group vars:  {out_grouped.var(axis=-1).round(3)}")

## 5. RMS Normalization

Used in LLaMA, GPT-NeoX, and many modern LLMs. Simpler than LayerNorm — no mean subtraction.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma, \quad \text{RMS}(x) = \sqrt{\frac{1}{D}\sum_{i=1}^{D} x_i^2}$$

**Why no bias (beta)?** Empirically, the mean-centering in LayerNorm is not needed; only the re-scaling matters.

In [ ]:
class RMSNorm:
    """Root Mean Square Normalization (no centering, no bias)."""
    
    def __init__(self, dim, eps=1e-8):
        self.gamma = np.ones(dim)
        self.eps = eps
        self.dgamma = np.zeros_like(self.gamma)
    
    def forward(self, x):
        self.x = x
        self.rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + self.eps)
        self.x_hat = x / self.rms
        return self.gamma * self.x_hat
    
    def backward(self, dout):
        D = self.x.shape[-1]
        x = self.x
        rms = self.rms
        
        self.dgamma = np.sum(dout * self.x_hat, axis=0)
        
        dx_hat = dout * self.gamma
        
        # d/dx (x / rms(x))
        dx = (dx_hat / rms) - (x / (D * rms ** 3)) * np.sum(dx_hat * x, axis=-1, keepdims=True)
        return dx
    
    def parameters(self):
        return [(self.gamma, self.dgamma)]


# Test and compare with LayerNorm
x = np.random.randn(4, 8)
rn = RMSNorm(8)
ln = LayerNorm(8)

out_rms = rn.forward(x)
out_ln = ln.forward(x)

print("RMSNorm output:")
print(f"  Per-sample RMS: {np.sqrt(np.mean(out_rms**2, axis=-1)).round(4)}")
print(f"  Per-sample mean: {out_rms.mean(axis=-1).round(4)} (NOT zero — no centering)")
print(f"\nLayerNorm output:")
print(f"  Per-sample mean: {out_ln.mean(axis=-1).round(4)} (zero — centered)")

## 6. Numerical Gradient Check for All Norms

In [ ]:
def grad_check_norm(norm_cls, name, x, init_kwargs, eps=1e-5):
    """Check backward pass against numerical gradients."""
    norm = norm_cls(**init_kwargs)
    out = norm.forward(x.copy())
    dout = np.random.randn(*out.shape)
    dx_analytical = norm.backward(dout)
    
    # Numerical gradient
    dx_numerical = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        x_plus = x.copy(); x_plus[idx] += eps
        x_minus = x.copy(); x_minus[idx] -= eps
        norm_p = norm_cls(**init_kwargs)
        norm_m = norm_cls(**init_kwargs)
        out_p = norm_p.forward(x_plus)
        out_m = norm_m.forward(x_minus)
        dx_numerical[idx] = np.sum((out_p - out_m) * dout) / (2 * eps)
    
    rel_err = np.max(np.abs(dx_analytical - dx_numerical) / 
                     np.maximum(np.abs(dx_analytical) + np.abs(dx_numerical), 1e-8))
    status = 'OK' if rel_err < 1e-4 else 'FAIL'
    print(f"{name:<15} max rel error: {rel_err:.2e}  {status}")

np.random.seed(42)
x = np.random.randn(4, 8)

print("Gradient checks (dx):")
print("-" * 45)
grad_check_norm(BatchNorm, 'BatchNorm', x, {'num_features': 8})
grad_check_norm(LayerNorm, 'LayerNorm', x, {'normalized_shape': 8})
grad_check_norm(GroupNorm, 'GroupNorm', x, {'num_groups': 2, 'num_channels': 8})
grad_check_norm(RMSNorm, 'RMSNorm', x, {'dim': 8})

## 7. Demo: Training With vs Without Normalization

In [ ]:
# Simple 3-layer network with optional normalization
def make_data(n=500):
    np.random.seed(42)
    X = np.random.randn(n, 2)
    y = (X[:, 0] ** 2 + X[:, 1] ** 2 > 1).astype(int)  # circle boundary
    return X, y

X_data, y_data = make_data()


def train_with_norm(use_norm=None, epochs=200, lr=0.1):
    """Train a 3-layer network with or without normalization."""
    np.random.seed(42)
    H = 32
    
    # Init weights
    W1 = np.random.randn(2, H) * np.sqrt(2.0 / 2)
    b1 = np.zeros((1, H))
    W2 = np.random.randn(H, H) * np.sqrt(2.0 / H)
    b2 = np.zeros((1, H))
    W3 = np.random.randn(H, 2) * np.sqrt(2.0 / H)
    b3 = np.zeros((1, 2))
    
    norms = []
    if use_norm == 'batchnorm':
        norms = [BatchNorm(H), BatchNorm(H)]
    elif use_norm == 'layernorm':
        norms = [LayerNorm(H), LayerNorm(H)]
    
    N = X_data.shape[0]
    losses = []
    
    for epoch in range(epochs):
        # Forward
        z1 = X_data @ W1 + b1
        if norms:
            z1 = norms[0].forward(z1)
        a1 = np.maximum(0, z1)
        
        z2 = a1 @ W2 + b2
        if norms:
            z2 = norms[1].forward(z2)
        a2 = np.maximum(0, z2)
        
        z3 = a2 @ W3 + b3
        
        # Softmax + CE
        shifted = z3 - np.max(z3, axis=1, keepdims=True)
        probs = np.exp(shifted) / np.sum(np.exp(shifted), axis=1, keepdims=True)
        loss = np.mean(-np.log(probs[np.arange(N), y_data] + 1e-12))
        losses.append(loss)
        
        # Backward
        dz3 = probs.copy()
        dz3[np.arange(N), y_data] -= 1.0
        dz3 /= N
        
        dW3 = a2.T @ dz3
        db3 = np.sum(dz3, axis=0, keepdims=True)
        da2 = dz3 @ W3.T
        
        dz2 = da2 * (z2 > 0).astype(float)
        if norms:
            dz2 = norms[1].backward(dz2)
        dW2 = a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)
        da1 = dz2 @ W2.T
        
        dz1 = da1 * (z1 > 0).astype(float)
        if norms:
            dz1 = norms[0].backward(dz1)
        dW1 = X_data.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)
        
        # Update
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
        W3 -= lr * dW3; b3 -= lr * db3
        
        if norms:
            for norm in norms:
                for p, g in norm.parameters():
                    p -= lr * g
    
    return losses


# Train with different normalizations
losses_none = train_with_norm(use_norm=None)
losses_bn = train_with_norm(use_norm='batchnorm')
losses_ln = train_with_norm(use_norm='layernorm')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_none, label='No normalization', linewidth=2)
ax.plot(losses_bn, label='BatchNorm', linewidth=2, linestyle='--')
ax.plot(losses_ln, label='LayerNorm', linewidth=2, linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Convergence Speed: With vs Without Normalization')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. BatchNorm Inference Mode

In [ ]:
# Show how running statistics evolve during training
np.random.seed(42)
bn = BatchNorm(4, momentum=0.1)

running_means = [bn.running_mean.copy()]
running_vars = [bn.running_var.copy()]

for step in range(100):
    # Simulate training batches with mean=3, std=2
    x = np.random.randn(32, 4) * 2 + 3
    _ = bn.forward(x)
    running_means.append(bn.running_mean.copy())
    running_vars.append(bn.running_var.copy())

running_means = np.array(running_means).squeeze()
running_vars = np.array(running_vars).squeeze()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(running_means[:, 0], label='Feature 0')
ax1.axhline(3.0, color='r', linestyle='--', label='True mean')
ax1.set_xlabel('Step'); ax1.set_ylabel('Running Mean')
ax1.set_title('Running Mean Converges to True Mean')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(running_vars[:, 0], label='Feature 0')
ax2.axhline(4.0, color='r', linestyle='--', label='True variance')
ax2.set_xlabel('Step'); ax2.set_ylabel('Running Var')
ax2.set_title('Running Var Converges to True Variance')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test inference mode
bn.training = False
x_test = np.random.randn(1, 4) * 2 + 3
out_test = bn.forward(x_test)
print(f"\nInference mode uses running stats (no batch dependency).")
print(f"Single sample input works: shape={x_test.shape} -> output shape={out_test.shape}")

## 9. RMSNorm vs LayerNorm — Comparison

In [ ]:
# Compare computational cost and output distribution
np.random.seed(42)
D = 512  # typical hidden dim
x = np.random.randn(32, D) * 5 + 2  # skewed input

ln = LayerNorm(D)
rn = RMSNorm(D)

out_ln = ln.forward(x)
out_rn = rn.forward(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Input distribution
axes[0].hist(x.ravel(), bins=50, alpha=0.7, density=True)
axes[0].set_title(f'Input (mean={x.mean():.2f}, std={x.std():.2f})')
axes[0].set_xlabel('Value')

# LayerNorm output
axes[1].hist(out_ln.ravel(), bins=50, alpha=0.7, density=True, color='orange')
axes[1].set_title(f'LayerNorm (mean={out_ln.mean():.4f}, std={out_ln.std():.2f})')
axes[1].set_xlabel('Value')

# RMSNorm output
axes[2].hist(out_rn.ravel(), bins=50, alpha=0.7, density=True, color='green')
axes[2].set_title(f'RMSNorm (mean={out_rn.mean():.4f}, std={out_rn.std():.2f})')
axes[2].set_xlabel('Value')

plt.tight_layout()
plt.show()

print("Key difference: RMSNorm preserves the mean (no centering), LayerNorm centers to ~0.")
print("RMSNorm advantage: simpler computation, fewer parameters (no beta), works equally well for LLMs.")

## 10. PyTorch Comparison

In [ ]:
try:
    import torch
    import torch.nn as nn
    
    np.random.seed(42)
    x_np = np.random.randn(4, 8).astype(np.float32)
    
    # LayerNorm comparison
    ln_ours = LayerNorm(8)
    out_ours = ln_ours.forward(x_np.astype(np.float64))
    
    ln_pt = nn.LayerNorm(8, elementwise_affine=True)
    # Set same gamma/beta
    with torch.no_grad():
        ln_pt.weight.fill_(1.0)
        ln_pt.bias.fill_(0.0)
    x_pt = torch.tensor(x_np, requires_grad=True)
    out_pt = ln_pt(x_pt)
    
    print("LayerNorm output comparison:")
    print(f"  Max abs diff: {np.max(np.abs(out_ours - out_pt.detach().numpy())):.2e}")
    print(f"  Match: {np.allclose(out_ours, out_pt.detach().numpy().astype(np.float64), atol=1e-5)}")
    
    # Check backward
    dout = np.random.randn(4, 8).astype(np.float64)
    dx_ours = ln_ours.backward(dout)
    
    out_pt.backward(torch.tensor(dout.astype(np.float32)))
    dx_pt = x_pt.grad.numpy()
    
    print(f"\nLayerNorm gradient comparison:")
    print(f"  Max abs diff: {np.max(np.abs(dx_ours - dx_pt)):.2e}")
    
except ImportError:
    print("PyTorch not available — skipping comparison.")

## Interview Quick Reference

**Q: Why LayerNorm in transformers, not BatchNorm?**  
1. BatchNorm depends on batch statistics => breaks with variable-length sequences (padding), small batches, and inference on single samples
2. LayerNorm normalizes per sample => independent of batch size, works naturally with sequences
3. In transformers, each token is processed independently through LN

**Q: What does BatchNorm do during inference?**  
Uses exponential moving average of training batch means/variances (running_mean, running_var). No dependence on current batch. This is why model.eval() is critical.

**Q: RMSNorm vs LayerNorm?**  
RMSNorm drops the mean subtraction and bias term. Empirically works as well as LayerNorm for LLMs but is ~10% faster (fewer operations). Used in LLaMA, Mistral, etc.

**Q: Why does normalization help training?**  
1. Smooths the loss landscape (fewer sharp minima)
2. Reduces internal covariate shift (debated but original motivation)
3. Allows higher learning rates without divergence
4. Acts as mild regularizer (batch-dependent noise in BN)

**Q: BatchNorm breaks with small batches — why?**  
Batch statistics become noisy/unreliable with small N. Variance estimate has high variance itself. GroupNorm is the fix for small-batch settings (used in detection models).

---
*Next: 06_regularization.ipynb — dropout, weight decay, and more*